In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, LongType, IntegerType, ArrayType
)
from delta.tables import DeltaTable
from datetime import datetime
from pyspark.sql.functions import col

In [0]:
schema = StructType([
    StructField("S_No", LongType(), True),
    StructField("DataMart", StringType(), True),
    StructField("Target_Catalog", StringType(), True),
    StructField("Target_Schema", StringType(), True),
    StructField("Target_Table_Name", StringType(), True),
    StructField("Source", StringType(), True),
    StructField("Source_Path", StringType(), True),
    StructField("Source_File_Name", StringType(), True),
    StructField("Source_File_Format", StringType(), True),
    StructField("Primary_Fields", ArrayType(StringType()), True),
    StructField("Date_Field", StringType(), True),
    StructField("File_Type", StringType(), True),
    StructField("File_Count", IntegerType(), True),
    StructField("Is_Active", LongType(), True)
])


In [0]:
data = [
    (
        1,
        "healthcare_test",
        "iics",
        "casemix",
        "healthcare_ingestion_test",
        "s3",
        "s3://cittabase-s3-test/Databricks-poc/moh_test/ingestion_test/",
        "case_data_003.csv",
        "csv",
        ['CASE_SURR_ID'],
        "DH_UPDATED_DATE",
        "data",
        10000,
        1
    ),
    (
        2,
        "healthcare_test",
        "iics",
        "casemix",
        "healthcare_ingestion_test",
        "s3",
        "s3://cittabase-s3-test/Databricks-poc/moh_test/ingestion_test/",
        "case_data_004.csv",
        "csv",
        ['CASE_SURR_ID'],
        "DH_UPDATED_DATE",
        "data",
        10000,
        1
    ),
    (
        3,
        "healthcare_test",
        "iics",
        "casemix",
        "healthcare_merge_test",
        "s3",
        "s3://cittabase-s3-test/Databricks-poc/moh_test/merge_test/",
        "case_data_003.csv",
        "csv",
        ['CASE_SURR_ID'],
        "DH_UPDATED_DATE",
        "data",
        10000,
        1
    ),
    (
        4,
        "healthcare_test",
        "iics",
        "casemix",
        "healthcare_merge_test",
        "s3",
        "s3://cittabase-s3-test/Databricks-poc/moh_test/merge_test/",
        "case_data_004.csv",
        "csv",
        ['CASE_SURR_ID'],
        "DH_UPDATED_DATE",
        "data",
        10000,
        1
    ),
    (
        5,
        "healthcare_test",
        "iics",
        "casemix",
        "healthcare_merge_test",
        "s3",
        "s3://cittabase-s3-test/Databricks-poc/moh_test/merge_test/",
        "case_data_005.csv",
        "csv",
        ['CASE_SURR_ID'],
        "DH_UPDATED_DATE",
        "data",
        10000,
        1
    ),
    (
        6,
        "healthcare_test",
        "iics",
        "casemix",
        "healthcare_error_test",
        "s3",
        "s3://cittabase-s3-test/Databricks-poc/moh_test/error_test/",
        "case_data_003.csv",
        "csv",
        ['CASE_SURR_ID'],
        "DH_UPDATED_DATE",
        "data",
        10000,
        1
    ),
    (
        7,
        "healthcare_test",
        "iics",
        "casemix",
        "healthcare_error_test",
        "s3",
        "s3://cittabase-s3-test/Databricks-poc/moh_test/error_test/",
        "case_data_004.csv",
        "csv",
        ['CASE_SURR_ID'],
        "DH_UPDATED_DATE",
        "data",
        10000,
        1
    ),
]

In [0]:
data = [
    (
        1,
        "healthcare_test",
        "iics",
        "casemix",
        "healthcare_test",
        "s3",
        "s3://cittabase-s3-test/Databricks-poc/iics_test/",
        "case_data_003.csv",
        "csv",
        ['CASE_SURR_ID'],
        "DH_UPDATED_DATE",
        "data",
        10000,
        1
    ),
    (
        2,
        "healthcare_test",
        "iics",
        "casemix",
        "healthcare_test",
        "s3",
        "s3://cittabase-s3-test/Databricks-poc/iics_test/",
        "case_data_004.csv",
        "csv",
        ['CASE_SURR_ID'],
        "DH_UPDATED_DATE",
        "data",
        10000,
        1
    )
]

In [0]:
df_control_table_creation = spark.createDataFrame(data, schema)

In [0]:
display(df_control_table_creation)

In [0]:
df_control_table_creation.write.format("delta").mode("overwrite").saveAsTable("iics.casemix.csv_file_control_test")

In [0]:
%sql
SELECT * FROM iics.casemix.csv_file_control_v1 

In [0]:
%sql
SELECT * FROM iics.casemix.csv_file_control_test -- WHERE S_No=2255;

In [0]:
%sql
drop table if exists iics.casemix.csv_file_logs_test;

In [0]:
%sql
select * from iics.casemix.healthcare_test

In [0]:
df_source = (spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv("s3://cittabase-s3-test/Databricks-poc/iics_test/case_data_005.csv")
    )

In [0]:
df_source = df_source.withColumn("loaded_at", F.lit(datetime.now()))

In [0]:
df_source.count()

In [0]:
display(df_source)

In [0]:
%sql
-- before merge
SELECT * FROM iics.casemix.healthcare_test;

In [0]:
target_table = DeltaTable.forName(spark, "iics.casemix.healthcare_test")


In [0]:
target_table.alias("tgt") \
    .merge(
        df_source.alias("src"),
        "tgt.CASE_SURR_ID = src.CASE_SURR_ID"
    ) \
    .whenMatchedUpdateAll(
        condition = "src.DH_UPDATED_DATE > tgt.DH_UPDATED_DATE",
    ) \
    .whenNotMatchedInsertAll() \
    .execute()

In [0]:
%sql
-- before merge
SELECT COUNT(*) FROM iics.casemix.healthcare_test; -- 20000

In [0]:
%sql
-- before merge
select * from iics.casemix.healthcare_test where CASE_SURR_ID IN ('10001','10002','10003','10004','10005','10006','10007','10008','10009','10010')

In [0]:
%sql
-- after merge
select S_No,count(*) as row_count from iics.casemix.healthcare_merge_test
group by S_No
;

In [0]:
%sql
-- after merge
select S_No,count(*) as row_count from iics.casemix.healthcare_error_test
group by S_No
;

In [0]:
%sql
-- after merge
select * from iics.casemix.healthcare_test where CASE_SURR_ID IN ('10001','10002','10003','10004','10005','10006','10007','10008','10009','10010')

In [0]:
%sql
DROP TABLE IF EXISTS iics.casemix.csv_file_logs_test

In [0]:
%sql
select * from iics.casemix.healthcare_merge_test

In [0]:
%sql
select * from iics.casemix.csv_file_logs_test

In [0]:
%sql
SELECT COUNT(*) FROM iics.casemix.healthcare_error_test

In [0]:
df_json = spark.read.format("json").option("multiline", "true").load("/Volumes/iics/casemix/history/carereport_sample.json")

In [0]:
display(df_json)

In [0]:
df_json.write.mode("overwrite").saveAsTable("iics.casemix.carereport_sample")

In [0]:
%sql
SELECT * FROM iics.casemix.carereport_sample

In [0]:
%sql
SELECT * FROM read_files(
  '/Volumes/iics/casemix/history/carereport_sample.json',
  format => 'json',
  multiLine => true)

In [0]:
%sql
CREATE TABLE iics.casemix.carereport_sample
USING JSON
OPTIONS (
  path 's3://moh-s3-dev-bucket/Volumes/iics/casemix/history/carereport_sample.json',
  multiLine 'true'
);

In [0]:
%sql
SELECT * FROM reviews_json_table;

In [0]:

df = (
    spark.read.text("/Volumes/iics/casemix/history/carereport_sample.json")
    .select(col("value").alias("json_data").cast("variant"))
)



In [0]:
display(df)

In [0]:
df.dtypes

In [0]:
cols = ["id","name","designation","is_active"]
sample_data = [(1,"sulthan","data_engineer",1)]

In [0]:
spark.createDataFrame(data=sample_data,schema=cols).write.mode("overwrite").option("comment","TEST").saveAsTable("iics.casemix.comments_tags_test")

In [0]:
%sql
SELECT * FROM iics.information_schema.tables WHERE table_name='comments_tags_test'

In [0]:
%sql
SELECT * FROM iics.information_schema.table_tags WHERE table_name='comments_tags_test'

In [0]:
spark.sql(f"""
ALTER TABLE iics.casemix.comments_tags_test
SET TAGS (
    'layer'='bronze',
    'user_group'='data_engineer'
)
""")

In [0]:
%sql
SELECT current_user();

In [0]:
%sql
SHOW GROUPS;

In [0]:
%sql
CREATE OR REPLACE FUNCTION iics.casemix.mask_name(value STRING)
RETURNS STRING
RETURN 
    CASE
        WHEN current_user() = 'mohamedsulthan.m@cittabase.com' THEN value
    ELSE '*****'
END;

In [0]:
%sql
SELECT * FROM iics.casemix.comments_tags_test

In [0]:
%sql
SELECT * FROM iics.casemix.healthcare_ingestion_test_1 LIMIT 2

In [0]:
%sql
SELECT cc.Source_File_Name,COUNT(*) as record_count FROM iics.casemix.healthcare_ingestion_test_1 as ht LEFT JOIN iics.casemix.csv_file_control_test as cc on ht.S_No=cc.S_No
group by cc.Source_File_Name;

In [0]:
%sql
SELECT * FROM iics.casemix.csv_file_control_test where Is_Active=1

In [0]:
%sql
SELECT * FROM iics.casemix.healthcare_ingestion_test_1

In [0]:
%sql
SELECT * FROM iics.casemix.healthcare_ingestion_test_1

In [0]:
from pyspark.sql import functions as F 

In [0]:
df = spark.read.table("iics.casemix.healthcare_ingestion_test_1")

In [0]:
df = df.withColumn(
    "hash_key",
    F.sha2(
        F.concat_ws(
            "||",
            F.col("PATIENT_SURR_ID"),
            F.col("CASE_SURR_ID"),
            F.col("CASE_TYPE"),
            F.col("PATIENT_AC_NO")
        ),
        256
    )
)

In [0]:
null_value = "<NULL>"

df = df.withColumn(
    "hash_key",
    F.sha2(
        F.concat_ws(
            F.coalesce(F.col("PATIENT_SURR_ID").cast("string"), F.lit(null_value)),
            F.coalesce(F.col("CASE_SURR_ID").cast("string"), F.lit(null_value)),
            F.coalesce(F.col("CASE_TYPE").cast("string"), F.lit(null_value)),
            F.coalesce(F.col("PATIENT_AC_NO").cast("string"), F.lit(null_value)),
        ),
        256,
    ),
)

In [0]:
display(df)

In [0]:
df = df.withColumn(
    "hash_key_md5",
    F.md5(
        F.concat_ws(
            "||",
            F.col("PATIENT_SURR_ID"),
            F.col("CASE_SURR_ID"),
            F.col("CASE_TYPE"),
            F.col("PATIENT_AC_NO")
        )
    )
)